# R1: Shared structure of public concerns across technologies

**Finding being tested**: UK public dialogue reports commissioned across two
decades and 11 technologies converge on a shared *procedural grammar of
legitimacy* — concern about how technology is governed, who participates in
decisions, and how information flows. The substance of the technology matters
less than the procedural questions surrounding it.

This notebook loads pre-computed artifacts from `01_processing.ipynb` and
`01a_clustering.ipynb` (concern and benefit phrase embeddings, KMeans cluster
assignments at k=75, cluster labels, and framing lens mappings). No API calls
are made here. The analysis proceeds entirely from the saved outputs.

**Sections in this notebook**:
1. Framing lens prevalence across all technologies (aggregate concern and benefit profiles)
2. Dendrogram validation of the lens scheme (hierarchical clustering vs LLM-assigned lenses)
3. Cross-cutting threshold: which clusters appear across all 11 technologies
4. Stable-core scatterplot (Figure 1): entropy × frequency to identify the shared core
5. Benefit parallel: the same cross-cutting structure on the benefit side

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du
from pub_dialogue.utils import show_status, show_complete, AccessStage, AddressStage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- CIP-0010: stage objects centralise all path / parameter constants ---
_access  = AccessStage()
_address = AddressStage(access=_access)
OUTPUT_FOLDER     = _access.output_folder
CHECKPOINT_FOLDER = _access.checkpoint_folder
TECH_COL          = _address.tech_col

a = _access.load_artifacts()

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

# extracted_concerns/benefits.csv are saved before the technology_meta merge
# in 01_processing.ipynb, so we re-apply it here if needed.
if "technology_meta" not in concerns_df.columns:
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

# Convenience aliases: uppercase names used by analysis cells
FRAMING_LENS_MAPPINGS        = framing_lens_mappings
BENEFIT_FRAMING_LENS_MAPPINGS = benefit_framing_lens_mappings

# Aliases for variable names used in analysis cells
centroids            = concern_centroids
cluster_labels_dict        = cluster_labels
benefit_cluster_labels_dict = benefit_cluster_labels

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

## Framing lens prevalence (concerns)

Aggregate the 17,596 concern phrases by framing lens to see the overall
distribution across the corpus. The 10 concern lenses are conceptual
groupings of the 75 KMeans clusters assigned by the LLM in
`01a_clustering.ipynb`. This gives the baseline: before splitting AI from
non-AI, how are concerns distributed across the framing space?

In [ ]:
# @title Compute framing lens prevalence

show_status("Computing framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Concerns': v}
    for k, v in lens_prevalence.items()
]).sort_values('Concerns', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Concerns'], color='steelblue')
ax.set_xlabel('Number of Concern Phrases')
ax.set_title('Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Concerns']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Framing lens prevalence computed")

## Dendrogram validation of the concern framing-lens scheme

Hierarchical clustering of the 75 cluster centroids provides a data-driven
check on the LLM-assigned framing lenses. The Adjusted Rand Index (ARI)
between the dendrogram partition and the lens scheme is **0.185** for
concerns — indicating that the lens scheme is an interpretively useful
organisation but does not carve at strong joints in the data structure.

Some lenses (notably Privacy/Consent & Data) validate cleanly against the
data-driven hierarchy. Others are looser conceptual groupings. Findings
reported at the lens level are hedged accordingly: the scheme organises the
data into theoretically meaningful groups; it is not a strong empirical
partition.

In [ ]:
# @title Hierarchical structure of concern clusters (dendrogram)
# Validates the framing-lens scheme by showing whether concern clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing concern-cluster dendrogram...")

# L2-normalise centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(centroids, axis=1, keepdims=True)
centroids_normed = centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and average linkage
distances = pdist(centroids_normed, metric="cosine")
Z = linkage(distances, method="average")

# Build leaf labels from existing cluster labels
cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in cluster_labels_dict.items()
}
leaf_labels = [cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(centroids_normed))]

# Build cluster->lens lookup
cluster_to_lens = {}
for lens_name, info in FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names = list(FRAMING_LENS_MAPPINGS.keys())
lens_colours = {name: cm.tab20(i / max(len(lens_names), 1))
                for i, name in enumerate(lens_names)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours.get(lens, "black"))

ax.set_xlabel("Cosine distance between concern cluster centroids")
ax.set_title("Hierarchical structure of concern clusters\n(leaf colour = LLM-assigned framing lens)")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: ARI between LLM lens assignment and dendrogram cut
from sklearn.metrics import adjusted_rand_score

n_lenses = len(FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses, criterion="maxclust")
lens_assignment = np.array([
    lens_names.index(cluster_to_lens.get(i, lens_names[0]))
    if cluster_to_lens.get(i) in lens_names else -1
    for i in range(len(centroids_normed))
])
mask = lens_assignment >= 0
ari = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of concern lenses (LLM-derived): {n_lenses}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses} groups: {ari:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Concern-cluster dendrogram complete.")

## Cross-cutting threshold: normalised entropy ≥ 0.5

A concern cluster is **cross-cutting** if its normalised entropy across
technologies is ≥ 0.5. Normalised entropy is the Shannon entropy of the
technology distribution within a cluster, divided by log(N_technologies),
so it runs from 0 (one technology only) to 1 (perfectly uniform across
all technologies). A threshold of 0.5 means the cluster's concern phrases
are spread across at least several technologies in a reasonably balanced way.

**Key result**: 62 of 75 concern clusters (83%) are cross-cutting, and these
hold 14,771 of 17,596 concern phrases (83.9%). Concern about emerging
technology is predominantly shared, not technology-specific.

In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

import numpy as np

TECH_COL = "technology_meta"

# v16-fix: compute normalised entropy locally rather than relying on a global
# called `normalized_entropy` (which is a function in this notebook, not a dict).
# This mirrors the same fix in the stable-core figure cell.
n_techs_for_norm = concerns_df[TECH_COL].nunique()
max_ent_for_norm = np.log(n_techs_for_norm) if n_techs_for_norm > 1 else 1.0
_norm_ent_for_xcut = {cid: (e / max_ent_for_norm) for cid, e in cluster_entropy.items()}

CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids = {
    cid for cid, ent in _norm_ent_for_xcut.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}

print(f"Cross-cutting clusters: {len(cross_cutting_cluster_ids)} of {len(cluster_entropy)} "
      f"(normalised entropy >= {CROSS_CUTTING_ENTROPY_THRESHOLD})")

## Top cross-cutting concern clusters (Table 1)

The 20 largest cross-cutting clusters account for **40.1% of all 17,596
concern phrases** across 65 documents and 11 technologies. These clusters
are dominated by procedural and structural concerns:

- Governance, oversight, and accountability
- Transparency in decision-making and information flows
- Public engagement and participation in decisions
- Safety and harm distribution
- Two data-specific clusters (Misuse of personal data; Privacy and data security)

The two data-specific clusters in the top 20 are the first signal of AI's
informational footprint — but they sit *within* a shared procedural grammar
that runs across all 11 technologies. See `03_ai_distinctiveness.ipynb` for
the AI-specific analysis.

In [ ]:
# @title Shared concerns across technologies (document-weighted)

show_status("Computing shared concern structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Salience matrix: rows=technology, cols=cluster_id, values=proportion of concerns
salience_rows = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask, "cluster_id"].value_counts()
    salience_rows[tech] = (counts / tech_total).to_dict()

salience_df = pd.DataFrame(salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
global_cluster_prevalence = concerns_df["cluster_id"].value_counts(normalize=True)

# Cross-cutting metric (entropy over technologies) computed earlier as cluster_entropy
cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(cluster_entropy.keys()),
            "tech_entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
            "global_prevalence": [global_cluster_prevalence.get(c, 0) for c in cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared concerns = high prevalence + high entropy
shared_concerns = cluster_meta_df.head(20)

print("Top shared concern clusters (high prevalence + cross-technology spread):")
display(shared_concerns.head(12))

# Quick bar chart (prevalence)
plt.figure(figsize=(10, 5))
shared_concerns.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared concerns across technologies (top 20 clusters by prevalence × spread)")
plt.xlabel("Share of all extracted concern phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_concerns_top20.png")
plt.show()

shared_concerns.reset_index().to_csv(OUTPUT_FOLDER / "shared_concerns_top20.csv", index=False)

show_complete("Shared concerns computed")


## Framing lens prevalence by technology (document-weighted)

How prominent is each framing lens within each technology's concern profile?
This computation produces a document-weighted salience matrix (technologies ×
lenses) that underpins the AI-vs-non-AI comparison in
`03_ai_distinctiveness.ipynb`. Here we look at the aggregate picture across
all technologies to confirm the dominance of cross-cutting lenses
(Governance & Accountability, Public Participation, Safety/Health/Environment)
before isolating AI's distinctive deviation.

In [ ]:
# @title Shared framings across technologies (document-weighted)

from scipy.stats import entropy

show_status("Computing shared framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
lens_stats = []
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = concerns_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()  # document-weighted (each concern phrase counts equally)

    # Technology distribution within lens
    tech_counts = concerns_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_concerns_in_lens': int(lens_mask.sum())
    })

lens_meta_df = (pd.DataFrame(lens_stats)
                .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared framings (high prevalence + cross-technology spread):")
display(lens_meta_df.head(12))

# Scatter: prevalence vs entropy
plt.figure(figsize=(7,5))
plt.scatter(lens_meta_df['tech_entropy'], lens_meta_df['overall_prevalence'])
for _, r in lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted concerns")
plt.title("Shared framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_framings_scatter.png")
plt.show()

lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_framings.csv", index=False)
show_complete("Shared framings computed")


## Figure 1: Stable-core scatterplot of the 75 concern clusters

**Axes**: x = normalised entropy (0 = technology-specific, 1 = perfectly
cross-cutting); y = number of concern phrases (frequency).

**Reading the plot**: clusters in the top-right quadrant (high entropy +
high frequency) form the *shared core* — concerns that are both prominent
and cross-cutting. These are labelled. Clusters in the bottom-left are
either rare or technology-specific.

The plot makes the R1 finding visually immediate: the shared core is large,
and its content is procedural (governance, transparency, public engagement,
oversight) with two data themes (Misuse of personal data; Privacy and data
security) already visible at the top.

In [ ]:
# @title Visualise the stable core of public concerns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "cluster_entropy" not in globals():
    raise NameError("cluster_entropy not found. Run the cluster entropy section first.")
if "concerns_df" not in globals():
    raise NameError("concerns_df not found. Run concern extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold (applied to NORMALISED entropy)

# ---- build cluster-level dataframe ----
# v16-fix: my earlier patch assumed `normalized_entropy` was a dict, but it's
# actually a function defined elsewhere in the notebook. Compute normalised
# entropy locally from the raw `cluster_entropy` dict instead.
n_techs = concerns_df[TECH_COL].nunique()
max_entropy = np.log(n_techs) if n_techs > 1 else 1.0
norm_ent_dict = {cid: (e / max_entropy) for cid, e in cluster_entropy.items()}

# cluster size = number of concern phrases
cluster_size = concerns_df["cluster_id"].value_counts()

df = (
    pd.DataFrame({
        "cluster_id": list(norm_ent_dict.keys()),
        "entropy": [norm_ent_dict[c] for c in norm_ent_dict.keys()],
    })
    .set_index("cluster_id")
)
df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

# cluster labels (if available)
if "CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)  # top 25% by frequency
stable_core = cross_cutting & (size >= size_thresh)

# clusters to annotate (most "core")
score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

# ---- plot ----
plt.figure(figsize=(10, 7))
ax = plt.gca()

# shaded stable-core region
core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)
ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)
plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Concerns")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_stable_core_scatter.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## PCA of the concern embedding space

Projects the 75 cluster centroids into 2D using PCA. Point size is
proportional to cluster frequency; colour indicates framing lens. This
supplementary visualisation shows whether lenses correspond to geometrically
distinct regions of the embedding space — informing the same ARI question as
the dendrogram but from a different angle.

In [ ]:
# @title Visualise the concern embedding space
# Clusters projected into 2D using PCA over technology-salience vectors.
# Point size ~ cluster frequency proxy; color ~ framing lens (if available).

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

# 1) Load tech x cluster salience (in-memory)
if "salience_df" not in globals():
    raise NameError("salience_df not found. Run Section 6.1a to compute salience_df first.")

tech_by_cluster = salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)

# Convert to clusters x technologies
cluster_features = tech_by_cluster.T  # rows=clusters, cols=technologies
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

# 2) Size proxy (use total salience mass across technologies)
size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# 3) Load framing lens mapping if available (optional)
lens_map = None
json_path = os.path.join(ARTIFACT_DIR, "framing_lens_mappings.json")
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        lens_map = json.load(f)
    # Normalize to {cluster_label: lens}
    if lens_map and all(isinstance(v, list) for v in lens_map.values()):
        inv = {}
        for lens, clist in lens_map.items():
            for c in clist:
                inv[str(c)] = str(lens)
        lens_map = inv
    else:
        lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

# Encode lenses to integer codes
unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# 4) PCA projection (standardize across technologies)
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

# 5) Plot
plt.figure(figsize=(10, 7))
ax = plt.gca()

# Use a categorical colormap; map integer codes -> colors
cmap = plt.get_cmap("tab20", len(unique_lenses))  # tab20 gives distinct-ish categories
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Concern space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Concern space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Concern Space of Public Technology Concerns\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

# Annotate top-N largest clusters (by size proxy)
topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

# Legend: show up to 12 most frequent lenses (or all if <= 12)
lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_concern_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


In [ ]:
# @title Visualise cross-technology concern distribution
show_status("Creating cross-technology concern heatmap...")
fig = _address.cross_technology_heatmap(salience_df, CLUSTER_LABELS, "concern", top_n=40)
fig.show()
show_complete("Cross-technology concern heatmap complete")

## Benefits side: framing lens prevalence

The same analysis repeated for the 18,346 benefit phrases. The 10 benefit
framing lenses mirror the concern-side structure conceptually, but the
distribution differs: medical and clinical benefit clusters are concentrated
in genetic technologies, and service-efficiency benefits cluster around AI.

**Key result**: **50 of 75 benefit clusters (67%) are cross-cutting**,
holding 13,190 of 18,346 benefit phrases (71.9%). The benefit side is
somewhat *less* cross-cutting than the concern side (83%) because
technology-specific benefits (medical/genetic; service-efficiency/AI) are
more salient than technology-specific concerns.

In [ ]:
# @title Compute benefit framing lens prevalence

show_status("Computing benefit framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Benefits': v}
    for k, v in lens_prevalence.items()
]).sort_values('Benefits', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Benefits'], color='steelblue')
ax.set_xlabel('Number of Benefit Phrases')
ax.set_title('Benefit Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Benefits']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Benefit framing lens prevalence computed")


## Dendrogram validation of the benefit framing-lens scheme

ARI between the data-driven dendrogram partition and the LLM-assigned benefit
lens scheme is **0.150** — slightly lower than the concern-side ARI of 0.185.
The benefit lens scheme is a useful conceptual organisation, not a strong
empirical partition. The same hedging applies as on the concern side.

In [ ]:
# @title Hierarchical structure of benefit clusters (dendrogram)
# Validates the framing-lens scheme by showing whether benefit clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing benefit-cluster dendrogram...")

# L2-normalise benefit centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(benefit_centroids, axis=1, keepdims=True)
benefit_centroids_normed = benefit_centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and Ward linkage
distances = pdist(benefit_centroids_normed, metric="cosine")
Z = linkage(distances, method="average")  # average linkage works well with cosine

# Build leaf labels using existing benefit cluster labels
benefit_cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in benefit_cluster_labels_dict.items()
}
leaf_labels = [benefit_cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(benefit_centroids_normed))]

# Build cluster->lens lookup so we can colour leaves by lens
benefit_cluster_to_lens = {}
for lens_name, info in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        benefit_cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names_b = list(BENEFIT_FRAMING_LENS_MAPPINGS.keys())
lens_colours_b = {name: cm.tab20(i / max(len(lens_names_b), 1))
                  for i, name in enumerate(lens_names_b)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in benefit_cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = benefit_cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours_b.get(lens, "black"))

ax.set_xlabel("Cosine distance between benefit cluster centroids")
ax.set_title("Hierarchical structure of benefit clusters\n(leaf colour = LLM-assigned framing lens)")

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours_b.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: cut the dendrogram at the same number of branches
# as the lens scheme, then compute Adjusted Rand Index against the lens
# assignment. If high, the lenses are well-supported by the data structure.
from sklearn.metrics import adjusted_rand_score

n_lenses_b = len(BENEFIT_FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses_b, criterion="maxclust")
lens_assignment = np.array([
    lens_names_b.index(benefit_cluster_to_lens.get(i, lens_names_b[0]))
    if benefit_cluster_to_lens.get(i) in lens_names_b else -1
    for i in range(len(benefit_centroids_normed))
])
mask = lens_assignment >= 0
ari_b = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of benefit lenses (LLM-derived): {n_lenses_b}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses_b} groups: {ari_b:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Benefit-cluster dendrogram complete.")

## Benefit cross-cutting clusters: setup

Apply the same normalised-entropy threshold (≥ 0.5) to the benefit clusters.
This identifies which benefit clusters span multiple technologies — the
benefit equivalent of the shared-core concern clusters above.

In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

TECH_COL = "technology_meta"

# Ensure benefits_df has technology_meta
if TECH_COL not in benefits_df.columns:
    benefits_df = benefits_df.merge(
        chunks_df[["chunk_id", TECH_COL]],
        on="chunk_id",
        how="left"
    )

CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids_benefits = {
    cid for cid, ent in normalized_entropy_benefits.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}


## Top cross-cutting benefit clusters

The shared benefit core is anchored by themes such as improved outcomes and
quality of life, better access to services, and enhanced safety — broadly
parallel to the governance and participation themes on the concern side.
AI's distinctive benefit profile (service efficiency, data privacy/control,
research quality) is visible here as an over-representation relative to the
non-AI technologies. That comparison is developed fully in
`03_ai_distinctiveness.ipynb`.

In [ ]:
# @title Shared benefits across technologies (document-weighted)

show_status("Computing shared benefit structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

benefit_salience_rows = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask, "cluster_id"].value_counts()
    benefit_salience_rows[tech] = (counts / tech_total).to_dict()

benefit_salience_df = pd.DataFrame(benefit_salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
benefit_global_cluster_prevalence = benefits_df["cluster_id"].value_counts(normalize=True)

benefit_cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(benefit_cluster_entropy.keys()),
            "tech_entropy": [benefit_cluster_entropy[c] for c in benefit_cluster_entropy.keys()],
            "global_prevalence": [benefit_global_cluster_prevalence.get(c, 0) for c in benefit_cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared benefits = high prevalence + high entropy
shared_benefits = benefit_cluster_meta_df.head(20)

print("Top shared benefit clusters (high prevalence + cross-technology spread):")
display(shared_benefits.head(12))

plt.figure(figsize=(10, 5))
shared_benefits.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared benefits across technologies (top 20 clusters by prevalence x spread)")
plt.xlabel("Share of all extracted benefit phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefits_top20.png")
plt.show()

shared_benefits.reset_index().to_csv(OUTPUT_FOLDER / "shared_benefits_top20.csv", index=False)

show_complete("Shared benefits computed")


## Benefit framing lens prevalence by technology (document-weighted)

Produces the technology × benefit-lens salience matrix that feeds into the
AI-vs-non-AI benefit comparison in `03_ai_distinctiveness.ipynb`.

In [ ]:
# @title Shared benefit framings across technologies (document-weighted)

from scipy.stats import entropy as scipy_entropy

show_status("Computing shared benefit framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
benefit_lens_stats = []
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()

    tech_counts = benefits_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(scipy_entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    benefit_lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_benefits_in_lens': int(lens_mask.sum())
    })

benefit_lens_meta_df = (pd.DataFrame(benefit_lens_stats)
                        .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared benefit framings (high prevalence + cross-technology spread):")
display(benefit_lens_meta_df.head(12))

plt.figure(figsize=(7,5))
plt.scatter(benefit_lens_meta_df['tech_entropy'], benefit_lens_meta_df['overall_prevalence'])
for _, r in benefit_lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted benefits")
plt.title("Shared benefit framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefit_framings_scatter.png")
plt.show()

benefit_lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_benefit_framings.csv", index=False)
show_complete("Shared benefit framings computed")


## Figure: Stable-core scatterplot of the 75 benefit clusters

The benefit-side equivalent of Figure 1. The shared core is smaller (67%
cross-cutting vs 83% on the concern side), reflecting that technology-specific
benefits — particularly medical/clinical benefits in genetic technologies and
service-efficiency benefits in AI — cluster more tightly than
technology-specific concerns.

In [ ]:
# @title Visualise the stable core of public technology benefits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "benefit_cluster_entropy" not in globals():
    raise NameError("benefit_cluster_entropy not found. Run cell 59 first.")
if "benefits_df" not in globals():
    raise NameError("benefits_df not found. Run benefit extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold

# ---- build cluster-level dataframe ----
cluster_size = benefits_df["cluster_id"].value_counts()

# Note: benefit_cluster_entropy stores RAW entropy. We need normalized entropy
# for the 0.5 threshold to be meaningful. Use normalized_entropy_benefits.
df = (
    pd.DataFrame({
        "cluster_id": list(normalized_entropy_benefits.keys()),
        "entropy": [normalized_entropy_benefits[c] for c in normalized_entropy_benefits.keys()],
    })
    .set_index("cluster_id")
)

df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

if "BENEFIT_CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: BENEFIT_CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)
stable_core = cross_cutting & (size >= size_thresh)

score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

plt.figure(figsize=(10, 7))
ax = plt.gca()

core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)

ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)

plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Benefits")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_stable_core_scatter.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()


## PCA of the benefit embedding space

Benefit-side equivalent of the concern embedding space visualisation above.

In [ ]:
# @title Visualise the benefit embedding space

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

if "benefit_salience_df" not in globals():
    raise NameError("benefit_salience_df not found. Run Section 6.1a (benefits) to compute it first.")

tech_by_cluster = benefit_salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)

# Convert to clusters x technologies
cluster_features = tech_by_cluster.T
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# Load benefit framing lens mapping if available
lens_map = None
if "BENEFIT_FRAMING_LENS_MAPPINGS" in globals():
    lens_map_raw = BENEFIT_FRAMING_LENS_MAPPINGS
    lens_map = {}
    for lens_name, data in lens_map_raw.items():
        for cid in data.get('cluster_ids', []):
            lens_map[str(cid)] = str(lens_name)
else:
    json_path = os.path.join(ARTIFACT_DIR, "benefit_framing_lens_mappings.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            lens_map = json.load(f)
        if lens_map and all(isinstance(v, list) for v in lens_map.values()):
            inv = {}
            for lens, clist in lens_map.items():
                for c in clist:
                    inv[str(c)] = str(lens)
            lens_map = inv
        else:
            lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# PCA projection
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

plt.figure(figsize=(10, 7))
ax = plt.gca()

cmap = plt.get_cmap("tab20", len(unique_lenses))
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Benefit space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Benefit space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Benefit Space of Public Technology Benefits\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_benefit_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


In [ ]:
# @title Visualise cross-technology benefit distribution
show_status("Creating cross-technology benefit heatmap...")
fig = _address.cross_technology_heatmap(benefit_salience_df, BENEFIT_CLUSTER_LABELS, "benefit", top_n=40)
fig.show()
show_complete("Cross-technology benefit heatmap complete")

## Volume table: phrase counts and paragraph incidence by technology

Reports the number of concern and benefit phrases extracted per technology,
and the proportion of paragraphs that yielded at least one phrase. This
provides the denominator context for interpreting AI's apparent volume
dominance (41 of 66 documents) and confirms that phrase yield rates are
broadly comparable across technologies.

In [ ]:
# @title Compare volume (phrase counts and paragraph incidence)
show_status("Computing volume comparison table...")

vol_c = _address.volume_table(concerns_df, "concern")
vol_b = _address.volume_table(benefits_df, "benefit")
vol = vol_c.join(vol_b, how="outer").fillna(0).astype(int)
vol.index.name = "technology / year"

display(vol)
show_complete("Volume comparison complete")

## Top clusters side by side (concerns and benefits)

A side-by-side view of the top concern and benefit clusters overall. This is
the summary table view of the shared structure: the cross-cutting concern
grammar (governance, transparency, oversight) alongside the cross-cutting
benefit grammar (improved outcomes, access, safety). The contrast between the
two sides, and AI's departure from the non-AI norm on both, is the subject
of `03_ai_distinctiveness.ipynb`.

In [ ]:
# @title Top clusters side-by-side (overall)
show_status("Computing top clusters table...")

top_c = _address.top_clusters(concerns_df, cluster_summary_df, "concern", n=10)
top_b = _address.top_clusters(benefits_df, benefit_cluster_summary_df, "benefit", n=10)
top_all = pd.concat([top_c, top_b], ignore_index=True)

display(top_all)
show_complete("Top clusters table complete")